In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to datetime (UTC) for internal computations (per rule 10),
# while keeping numeric Unix seconds for outputs.
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')  # keep numeric unix seconds
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'].astype('Int64'), unit='s', utc=True)

# Helper to build overview
def overview_df(df, filename):
    total = int(len(df))
    if total == 0:
        min_ts = None
        max_ts = None
    else:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
    return pd.DataFrame([{'file': filename, 'rows': total, 'min_ts': min_ts, 'max_ts': max_ts}])

# 1) metric.csv aggregates
metric_overview = overview_df(df_metric, 'metric.csv')

# Ensure numeric 'value'
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

metric_pairs = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count='count', p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan)
    .reset_index()
    .sort_values('count', ascending=False)
)

metric_pairs_top50 = metric_pairs.head(50)

# 2) trace.csv aggregates
trace_overview = overview_df(df_trace, 'trace.csv')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

trace_pairs = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(count='count', p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan)
    .reset_index()
    .sort_values('count', ascending=False)
)

trace_pairs_top50 = trace_pairs.head(50)

# 3) log.csv aggregates
log_overview = overview_df(df_log, 'log.csv')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

log_pairs = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(count='count', p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan)
    .reset_index()
    .sort_values('count', ascending=False)
)

log_pairs_top50 = log_pairs.head(50)

# 4) error_logs.csv aggregates
error_overview = overview_df(df_error, 'error_logs.csv')

# counts per cmdb_id
err_counts = (
    df_error
    .groupby('cmdb_id', dropna=False)['timestamp']
    .agg(count='count')
    .reset_index()
)

# earliest message per cmdb_id (earliest timestamp)
df_error_sorted = df_error.sort_values(['cmdb_id', 'timestamp'])
earliest_msgs = df_error_sorted.groupby('cmdb_id', dropna=False).first().reset_index()[['cmdb_id', 'timestamp', 'message']]
earliest_msgs = earliest_msgs.rename(columns={'timestamp': 'earliest_timestamp', 'message': 'sample_message'})

err_summary = err_counts.merge(earliest_msgs, on='cmdb_id', how='left').sort_values('count', ascending=False).head(20)

# Keep outputs compact: selected columns only
metric_overview, metric_pairs_top50[['cmdb_id','kpi_name','count','p95']], trace_overview, trace_pairs_top50[['cmdb_id','trace_name','count','p95']], log_overview, log_pairs_top50[['cmdb_id','log_name','count','p95']], error_overview, err_summary[['cmdb_id','count','earliest_timestamp','sample_message']]

```
Out[1]:
```
summary = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv: total rows = 2021, timestamp range = [1705652280, 1705653720]. "
    "Top (cmdb_id, kpi_name) pairs (by row count) include multiple carts KPIs (cpu, error, latency-50, latency-90, mem) "
    "and many other services (each top pair has ~25 rows). P95 values were computed per pair (see detailed table earlier).\n\n"
    "trace.csv: total rows = 0 (no trace telemetry in this dataset).\n\n"
    "log.csv: total rows = 452, timestamp range = [1705652280, 1705653720]. "
    "Top log sources include carts-db and catalogue (log.row_count and log.error_count), front-end (row_count), etc.; "
    "per-pair counts and P95 of log.value were computed (some row_count P95s are large, e.g. front-end row_count P95 ~1679.0).\n\n"
    "error_logs.csv: total rows = 0 (no error log messages present).\n\n"
    "Overall notes: metric and log files contain data for the same time window [1705652280, 1705653720] in Unix seconds (UTC). "
    "Trace and error log datasets are empty. For further drill-down, inspect the top metric and log pairs returned earlier."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  rows      min_ts      max_ts
0  metric.csv  2021  1705652280  1705653720,          cmdb_id    kpi_name  count           p95
0          carts         cpu     25  1.354622e+00
1          carts       error     25  0.000000e+00
2          carts  latency-50     25  2.945971e+00
3          carts  latency-90     25  9.878108e+00
4          carts         mem     25  2.231625e+08
..           ...         ...    ...           ...
46       payment         mem     25  4.461295e+06
53      rabbitmq         cpu     25  8.908487e-01
52  queue-master      socket     25  3.000000e+00
51  queue-master         mem     25  2.681719e+08
50  queue-master      diskio     25  4.089966e+04

[50 rows x 4 columns],         file  rows min_ts max_ts
0  trace.csv     0   None   None, Empty DataFrame
Columns: [cmdb_id, trace_name, count, p95]
Index: [],       file  rows      min_ts      max_ts
0  log.csv   452  1705652280  1705653720,        cmdb_id         log_name  count     p95
3     carts-db    log.row_count     25   555.2
2     carts-db  log.error_count     25     0.0
4    catalogue  log.error_count     25     0.0
5    catalogue    log.row_count     25   149.4
7    front-end    log.row_count     25  1679.0
..         ...              ...    ...     ...
1        carts    log.row_count     20    77.5
20     user-db  log.error_count     10     0.0
21     user-db    log.row_count     10     4.0
14  session-db  log.error_count      4     0.0
15  session-db    log.row_count      4     5.0

[22 rows x 4 columns],              file  rows min_ts max_ts
0  error_logs.csv     0   None   None, Empty DataFrame
Columns: [cmdb_id, count, earliest_timestamp, sample_message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse existing df_metric loaded earlier in the session
# Ensure timestamp and value types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Components of interest
components = ['carts', 'catalogue', 'orders', 'payment', 'user']
df_sel = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# Compute global P95 and full counts per (cmdb_id, kpi_name) using entire series (BEFORE filtering)
full_stats = (
    df_sel
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(full_count='count', global_p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size > 0 else np.nan)
    .reset_index()
)

# 2) Incident window (UTC) -> unix seconds
start_ts = int(pd.Timestamp('2024-01-19 08:15:41', tz='UTC').timestamp())
end_ts   = int(pd.Timestamp('2024-01-19 08:45:41', tz='UTC').timestamp())

# Filter rows to the incident window (inclusive)
# Ensure timestamp not null before comparison
df_window = df_sel[df_sel['timestamp'].notna() & (df_sel['timestamp'].astype('int64') >= start_ts) & (df_sel['timestamp'].astype('int64') <= end_ts)].copy()

# Merge global_p95 into window rows to evaluate exceedances
df_window = df_window.merge(full_stats[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')

# Create exceed boolean and exceed_ts for earliest exceed computation
df_window['exceed'] = df_window['value'] > df_window['global_p95']
df_window['exceed_ts'] = df_window['timestamp'].where(df_window['exceed'])

# Aggregate per pair in-window stats
window_stats = (
    df_window
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        in_window_count = ('value','count'),
        in_window_exceed_count = ('exceed','sum'),
        earliest_exceed_ts = ('exceed_ts','min'),
        in_window_max = ('value','max')
    )
    .reset_index()
)

# 3) Merge full stats with window stats, fill missing window groups with zeros/NA
final = full_stats.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

final['in_window_count'] = final['in_window_count'].fillna(0).astype(int)
final['in_window_exceed_count'] = final['in_window_exceed_count'].fillna(0).astype(int)
final['earliest_exceed_ts'] = final['earliest_exceed_ts'].where(final['earliest_exceed_ts'].notna(), pd.NA)
final['in_window_max'] = final['in_window_max'].where(final['in_window_max'].notna(), pd.NA)

# Compute deviation_ratio = (in_window_max - global_p95)/global_p95 with p95==0 handling
def compute_dev(row):
    p95 = row['global_p95']
    mv = row['in_window_max']
    if pd.isna(p95) or pd.isna(mv):
        return np.nan
    if p95 == 0:
        if mv > 0:
            return np.inf
        elif mv == 0:
            return 0.0
        else:
            return np.nan
    return float((mv - p95) / p95)

final['deviation_ratio'] = final.apply(compute_dev, axis=1)

# Select and order columns as requested
final_table = final[['cmdb_id','kpi_name','full_count','global_p95','in_window_count','in_window_exceed_count','earliest_exceed_ts','in_window_max','deviation_ratio']]

# Sort by in_window_exceed_count desc then deviation_ratio desc; limit to top 50
final_table_sorted = final_table.sort_values(by=['in_window_exceed_count','deviation_ratio'], ascending=[False, False]).head(50)

# Display compact result
final_table_sorted

```
Out[2]:
```
summary_rows = final_table_sorted  # reuse result from previous computation

total_pairs = int(summary_rows.shape[0])
incident_start_unix = int(pd.Timestamp('2024-01-19 08:15:41', tz='UTC').timestamp())
incident_end_unix = int(pd.Timestamp('2024-01-19 08:45:41', tz='UTC').timestamp())

top_exceeds = summary_rows[summary_rows['in_window_exceed_count'] > 0]\
    .sort_values(['in_window_exceed_count','deviation_ratio'], ascending=[False, False])\
    .head(5)

lines = []
lines.append(f"Metrics-based anomaly summary for components [carts, catalogue, orders, payment, user]")
lines.append(f"Incident window (UTC, unix seconds): {incident_start_unix} to {incident_end_unix}")
lines.append(f"Total (cmdb_id, kpi_name) pairs analyzed: {total_pairs}")
lines.append("")

if top_exceeds.shape[0] == 0:
    lines.append("No in-window values exceeded their global P95 for the examined pairs.")
else:
    lines.append("Top pairs with in-window exceedances (sorted by exceed count then deviation):")
    for _, r in top_exceeds.iterrows():
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        full_count = int(r['full_count'])
        p95 = r['global_p95']
        iw_count = int(r['in_window_count'])
        exceed_count = int(r['in_window_exceed_count'])
        earliest = int(r['earliest_exceed_ts']) if pd.notna(r['earliest_exceed_ts']) else "NA"
        iw_max = r['in_window_max'] if pd.notna(r['in_window_max']) else "NA"
        dev = r['deviation_ratio']
        lines.append(
            f"- {cmdb} | {kpi}: full_count={full_count}, global_p95={p95:.6g}, "
            f"in_window_count={iw_count}, in_window_exceed_count={exceed_count}, "
            f"earliest_exceed_ts={earliest}, in_window_max={iw_max}, deviation_ratio={dev if np.isinf(dev) else round(dev,6)}"
        )

summary = "\n".join(lines)
summary, top_exceeds.head(10)

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  full_count  global_p95  in_window_count  in_window_exceed_count  earliest_exceed_ts  in_window_max  deviation_ratio
3       carts  latency-90          25    9.878108               25                       2          1705652460      13.717732         0.388700
14  catalogue    workload          25    3.513053               25                       2          1705652280       4.645789         0.322436
33       user  latency-50          23    0.005630               23                       2          1705652400       0.007122         0.265087
25    payment  latency-50          23    0.003000               23                       2          1705652400       0.003651         0.217001
36       user      socket          25   26.773333               25                       2          1705652400      30.950000         0.156001
..        ...         ...         ...         ...              ...                     ...                 ...            ...              ...
13  catalogue      socket          25    7.000000               25                       0                <NA>       7.000000         0.000000
23    payment      diskio          25    0.000000               25                       0                <NA>       0.000000         0.000000
24    payment       error          25    0.000000               25                       0                <NA>       0.000000         0.000000
28    payment      socket          25    7.000000               25                       0                <NA>       7.000000         0.000000
32       user       error          25    0.000000               25                       0                <NA>       0.000000         0.000000

[38 rows x 9 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse df_metric from previous session; if not present, load it
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure types and add datetime for internal checks (timestamps remain numeric for outputs)
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'].astype('Int64'), unit='s', utc=True)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Specified pairs
pairs = [
  ('carts','latency-90'),
  ('catalogue','workload'),
  ('user','latency-50'),
  ('payment','latency-50'),
  ('user','socket')
]

# Compute global P95 and full_count for each specified pair using full series BEFORE filtering
pairs_df = pd.DataFrame(pairs, columns=['cmdb_id','kpi_name'])
full_stats = (
    df_metric
    .merge(pairs_df, on=['cmdb_id','kpi_name'], how='inner')
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(full_count='count', global_p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan)
    .reset_index()
)

# Incident window (UTC) -> unix seconds
start_ts = int(pd.Timestamp('2024-01-19 08:15:41', tz='UTC').timestamp())
end_ts   = int(pd.Timestamp('2024-01-19 08:45:41', tz='UTC').timestamp())

results = []

for _, row in full_stats.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['global_p95']
    full_count = int(row['full_count'])
    # Filter full metric rows for this pair into the incident window
    df_pair = df_metric[(df_metric['cmdb_id']==cmdb) & (df_metric['kpi_name']==kpi)].copy()
    df_window = df_pair[
        df_pair['timestamp'].notna() &
        (df_pair['timestamp'].astype('int64') >= start_ts) &
        (df_pair['timestamp'].astype('int64') <= end_ts)
    ].sort_values('timestamp').reset_index(drop=True)
    # Determine exceed boolean (if p95 is NaN, treat as no exceed)
    if pd.isna(p95):
        df_window['exceed'] = False
    else:
        df_window['exceed'] = df_window['value'] > p95
    # Identify consecutive runs: create run id by changes in exceed (fill first shift with False)
    if not df_window.empty:
        df_window['prev_exceed'] = df_window['exceed'].shift(1, fill_value=False)
        df_window['run_id'] = (df_window['exceed'] != df_window['prev_exceed']).cumsum()
        # Select only exceed runs and aggregate per run
        exceed_runs = df_window[df_window['exceed']].groupby('run_id').agg(
            segment_start_ts = ('timestamp','min'),
            segment_end_ts = ('timestamp','max'),
            segment_length_in_points = ('timestamp','count'),
            segment_max_value = ('value','max')
        ).reset_index(drop=True)
    else:
        exceed_runs = pd.DataFrame(columns=['segment_start_ts','segment_end_ts','segment_length_in_points','segment_max_value'])
    # If there are runs, take earliest run; otherwise produce a zero-segment row as specified
    if not exceed_runs.empty:
        seg = exceed_runs.sort_values('segment_start_ts').iloc[0]
        seg_start = int(seg['segment_start_ts'])
        seg_end = int(seg['segment_end_ts'])
        seg_len = int(seg['segment_length_in_points'])
        seg_max = float(seg['segment_max_value'])
        # compute deviation ratio with p95==0 handling
        if pd.isna(p95):
            dev = np.nan
        else:
            if p95 == 0:
                dev = np.inf if seg_max > 0 else 0.0
            else:
                dev = float((seg_max - p95) / p95)
    else:
        seg_start = pd.NA
        seg_end = pd.NA
        seg_len = 0
        seg_max = pd.NA
        dev = 0.0
    results.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': p95,
        'segment_start_ts': seg_start,
        'segment_end_ts': seg_end,
        'segment_length_in_points': seg_len,
        'segment_max_value': seg_max,
        'segment_max_deviation_ratio': dev,
        'full_count': full_count
    })

final_df = pd.DataFrame(results)

# Keep output compact: one (earliest) segment per pair, sorted by segment_start_ts ascending (NA at end), limit <=20
# Ensure numeric start_ts for sorting, treat NA as large
final_df['sort_key'] = final_df['segment_start_ts'].apply(lambda x: x if pd.notna(x) else 10**18)
final_df_sorted = final_df.sort_values('sort_key').drop(columns=['sort_key']).head(20)

# Display compact table with requested columns (timestamps as unix seconds or NA)
final_df_sorted[['cmdb_id','kpi_name','global_p95','segment_start_ts','segment_end_ts','segment_length_in_points','segment_max_value','segment_max_deviation_ratio']]

```
Out[3]:
```
Summary (one earliest exceedance segment per requested pair; timestamps are Unix seconds UTC):

- catalogue | workload: global P95 = 3.513053. Earliest exceedance segment 1705652280 → 1705652280 (1 point). Segment max = 4.645789. Deviation = 0.322436 (~32.2% above P95).

- payment | latency-50: global P95 = 0.003000. Earliest exceedance segment 1705652400 → 1705652400 (1 point). Segment max = 0.003651. Deviation = 0.217001 (~21.7% above P95).

- user | latency-50: global P95 = 0.005630. Earliest exceedance segment 1705652400 → 1705652400 (1 point). Segment max = 0.005675. Deviation = 0.008003 (~0.8% above P95).

- user | socket: global P95 = 26.773333. Earliest exceedance segment 1705652400 → 1705652460 (2 points). Segment max = 30.95. Deviation = 0.156001 (~15.6% above P95).

- carts | latency-90: global P95 = 9.878108. Earliest exceedance segment 1705652460 → 1705652460 (1 point). Segment max = 13.717732. Deviation = 0.388700 (~38.9% above P95).

All five pairs had in-window exceedance segments; values and deviations above show the magnitude of the anomalies (largest relative deviations seen for carts latency-90 and catalogue workload).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  global_p95  segment_start_ts  segment_end_ts  segment_length_in_points  segment_max_value  segment_max_deviation_ratio
1  catalogue    workload    3.513053        1705652280      1705652280                         1           4.645789                     0.322436
2    payment  latency-50    0.003000        1705652400      1705652400                         1           0.003651                     0.217001
3       user  latency-50    0.005630        1705652400      1705652400                         1           0.005675                     0.008003
4       user      socket   26.773333        1705652400      1705652460                         2          30.950000                     0.156001
0      carts  latency-90    9.878108        1705652460      1705652460                         1          13.717732                     0.388700```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse df_log from earlier session; if not present, load it
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure types and auxiliary datetime for internal checks (timestamps remain numeric in outputs)
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
df_log['timestamp_dt'] = pd.to_datetime(df_log['timestamp'].astype('Int64'), unit='s', utc=True)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# 1) Candidate cmdb_ids
candidates = ['carts','catalogue','orders','payment','user','carts-db','catalogue-db','user-db','front-end']

# 2) Compute global P95 and full_count per (cmdb_id, log_name) for candidates
df_candidates = df_log[df_log['cmdb_id'].isin(candidates)].copy()

full_log_stats = (
    df_candidates
    .groupby(['cmdb_id','log_name'], dropna=False)['value']
    .agg(full_count='count', global_p95=lambda x: float(x.dropna().quantile(0.95)) if x.dropna().size>0 else np.nan)
    .reset_index()
)

# Extract global P95 for log.row_count per cmdb_id (if exists)
p95_rowcount = full_log_stats[full_log_stats['log_name']=='log.row_count'][['cmdb_id','global_p95']].rename(columns={'global_p95':'p95_row_count'})

# 3) Filter to incident window
start_ts = int(pd.Timestamp('2024-01-19 08:15:41', tz='UTC').timestamp())
end_ts   = int(pd.Timestamp('2024-01-19 08:45:41', tz='UTC').timestamp())

df_window = df_candidates[
    df_candidates['timestamp'].notna() &
    (df_candidates['timestamp'].astype('int64') >= start_ts) &
    (df_candidates['timestamp'].astype('int64') <= end_ts)
].copy()

# Precompute helpful subsets
# rows with log.error_count > 0 in-window
err_rows = df_window[(df_window['log_name']=='log.error_count') & (df_window['value']>0)].copy()
# rows with log.row_count in-window
rowcount_rows = df_window[df_window['log_name']=='log.row_count'].copy()

# Prepare summary rows per cmdb_id
summary_list = []
for cmdb in candidates:
    sub = df_window[df_window['cmdb_id']==cmdb]
    in_window_row_count = int(sub.shape[0])
    # error_count > 0 count and earliest ts
    err_sub = err_rows[err_rows['cmdb_id']==cmdb]
    err_count = int(err_sub.shape[0])
    earliest_err_ts = int(err_sub['timestamp'].min()) if not err_sub.empty else pd.NA
    # row_count global p95 for this cmdb (if exists)
    p95_row = p95_rowcount[p95_rowcount['cmdb_id']==cmdb]
    if not p95_row.empty:
        p95_val = float(p95_row['p95_row_count'].iloc[0])
        # count of in-window rows where log.row_count > p95
        rc_sub = rowcount_rows[rowcount_rows['cmdb_id']==cmdb]
        rc_exceed_sub = rc_sub[rc_sub['value'] > p95_val]
        rc_exceed_count = int(rc_exceed_sub.shape[0])
        earliest_rc_exceed_ts = int(rc_exceed_sub['timestamp'].min()) if not rc_exceed_sub.empty else pd.NA
        max_in_window_row_count = float(rc_sub['value'].max()) if not rc_sub.empty else pd.NA
    else:
        p95_val = pd.NA
        rc_exceed_count = pd.NA
        earliest_rc_exceed_ts = pd.NA
        max_in_window_row_count = pd.NA
    summary_list.append({
        'cmdb_id': cmdb,
        'in_window_row_count': in_window_row_count,
        'num_in_window_error_count_gt0': err_count,
        'num_in_window_rowcount_gt_global_p95': rc_exceed_count,
        'earliest_ts_error_count_gt0': earliest_err_ts,
        'earliest_ts_rowcount_gt_p95': earliest_rc_exceed_ts,
        'max_in_window_log_row_count': max_in_window_row_count
    })

summary_table = pd.DataFrame(summary_list)

# Sorting: treat NA in num_in_window_rowcount_gt_global_p95 as lowest -> replace NA with -1 for sorting only
summary_table['rc_exceed_sort'] = summary_table['num_in_window_rowcount_gt_global_p95'].apply(lambda x: -1 if pd.isna(x) else int(x))
summary_table_sorted = summary_table.sort_values(by=['rc_exceed_sort','num_in_window_error_count_gt0'], ascending=[False, False]).drop(columns=['rc_exceed_sort'])

# Limit to top 20 rows
summary_table_sorted = summary_table_sorted.head(20)

# 5) Identify single cmdb_id with largest count of in-window row_count exceedances (tie-breaker: error_count)
# Use numeric sort key where NA -> -1
selection_df = summary_table.copy()
selection_df['rc_exceed_sort'] = selection_df['num_in_window_rowcount_gt_global_p95'].apply(lambda x: -1 if pd.isna(x) else int(x))
selection_df = selection_df.sort_values(by=['rc_exceed_sort','num_in_window_error_count_gt0'], ascending=[False, False]).reset_index(drop=True)

if selection_df.shape[0] == 0:
    top_cmdb = None
else:
    top_cmdb = selection_df.loc[0,'cmdb_id']

# For that cmdb_id, return up to 5 sample in-window log rows that are row_count exceedances (earliest 5)
sample_rows = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
if top_cmdb is not None:
    # find p95 for top_cmdb
    top_p95_row = p95_rowcount[p95_rowcount['cmdb_id']==top_cmdb]
    if not top_p95_row.empty:
        top_p95 = float(top_p95_row['p95_row_count'].iloc[0])
        candidate_exceed = rowcount_rows[(rowcount_rows['cmdb_id']==top_cmdb) & (rowcount_rows['value'] > top_p95)].sort_values('timestamp').head(5)
        if not candidate_exceed.empty:
            sample_rows = candidate_exceed[['timestamp','cmdb_id','log_name','value']].reset_index(drop=True)

# Final compact outputs: summary_table_sorted and sample_rows for top cmdb
summary_table_sorted, sample_rows

```
Out[4]:
```
Summary of log.csv analysis for the candidate components (timestamps shown as Unix seconds UTC):

Key findings
- No component had any in-window log.error_count > 0 (all num_in_window_error_count_gt0 = 0).
- Components with in-window log.row_count exceedances (count of rows where log.row_count > its global P95):
  - catalogue: 2 exceedances (earliest exceed ts = 1705652520), max in-window row_count = 151
  - user: 2 exceedances (earliest exceed ts = 1705652520), max in-window row_count = 536
  - carts-db: 2 exceedances (earliest exceed ts = 1705653120), max in-window row_count = 558
  - front-end: 2 exceedances (earliest exceed ts = 1705652520), max in-window row_count = 1730
  - carts: 1 exceedance (earliest exceed ts = 1705652880), max in-window row_count = 125
  - orders, payment, user-db, catalogue-db: 0 exceedances (catalogue-db had no rows in-window)
- Components with the largest max in-window row_count: front-end (1730), carts-db (558), user (536), catalogue (151), carts (125).

Top component and sample exceedance rows
- The component selected as top by exceedance count (tie-breaker applied) is: catalogue.
- Up to 5 earliest in-window log.row_count exceedance samples for catalogue:
  - (timestamp=1705652520, cmdb_id=catalogue, log_name=log.row_count, value=150)
  - (timestamp=1705653540, cmdb_id=catalogue, log_name=log.row_count, value=151)

Interpretation
- Several services show elevated log.row_count during the incident window (front-end, carts-db, user, catalogue, carts). These are counts/activity spikes rather than error-count spikes (no log.error_count > 0 observed).
- Front-end, carts-db and user show the largest absolute row_count values and may warrant further attention for load-related investigation.

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id  in_window_row_count  num_in_window_error_count_gt0 num_in_window_rowcount_gt_global_p95 earliest_ts_error_count_gt0 earliest_ts_rowcount_gt_p95 max_in_window_log_row_count
1     catalogue                   50                              0                                    2                        <NA>                  1705652520                       151.0
4          user                   46                              0                                    2                        <NA>                  1705652520                       536.0
5      carts-db                   50                              0                                    2                        <NA>                  1705653120                       558.0
8     front-end                   50                              0                                    2                        <NA>                  1705652520                      1730.0
0         carts                   40                              0                                    1                        <NA>                  1705652880                       125.0
2        orders                   46                              0                                    0                        <NA>                        <NA>                       110.0
3       payment                   50                              0                                    0                        <NA>                        <NA>                        95.0
7       user-db                   20                              0                                    0                        <NA>                        <NA>                         4.0
6  catalogue-db                    0                              0                                 <NA>                        <NA>                        <NA>                        <NA>,     timestamp    cmdb_id       log_name  value
0  1705652520  catalogue  log.row_count    150
1  1705653540  catalogue  log.row_count    151)```
```

In [ ]:
# In[5]:

# Reuse df_metric from earlier session; ensure types are correct
try:
    df_metric
except NameError:
    import pandas as pd
    df_metric = pd.read_csv("metric.csv")

import numpy as np
import pandas as pd

df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Specified pairs
pairs = [
    ('carts','latency-90'),
    ('catalogue','workload'),
    ('user','latency-50'),
    ('payment','latency-50'),
    ('user','socket')
]

# Incident window (UTC) -> unix seconds
start_ts = int(pd.Timestamp('2024-01-19 08:15:41', tz='UTC').timestamp())
end_ts   = int(pd.Timestamp('2024-01-19 08:45:41', tz='UTC').timestamp())

results = []
for cmdb,kpi in pairs:
    # 1) compute global P95 using full series BEFORE filtering
    df_pair_full = df_metric[(df_metric['cmdb_id']==cmdb) & (df_metric['kpi_name']==kpi)]
    if df_pair_full['value'].dropna().size > 0:
        global_p95 = float(df_pair_full['value'].dropna().quantile(0.95))
    else:
        global_p95 = np.nan

    # 2) filter to incident window
    df_pair_win = df_pair_full[
        df_pair_full['timestamp'].notna() &
        (df_pair_full['timestamp'].astype('int64') >= start_ts) &
        (df_pair_full['timestamp'].astype('int64') <= end_ts)
    ].sort_values('timestamp').reset_index(drop=True)

    # mark exceedances
    if pd.isna(global_p95):
        df_pair_win['is_exceed'] = False
    else:
        df_pair_win['is_exceed'] = df_pair_win['value'] > global_p95

    # 3) list of in-window exceed timestamps (earliest 20)
    exceed_ts_list = df_pair_win.loc[df_pair_win['is_exceed'], 'timestamp'].astype('Int64').tolist()
    exceed_ts_list = exceed_ts_list[:20]
    in_window_exceed_count = len(exceed_ts_list)

    # find consecutive exceed segments among the in-window ordered rows
    if df_pair_win.empty or in_window_exceed_count == 0:
        longest_len = 0
        earliest_start_of_longest = pd.NA
        segment_max_value = pd.NA
        segment_max_dev = 0.0
    else:
        # create run ids based on changes in is_exceed
        df_pair_win['prev_exceed'] = df_pair_win['is_exceed'].shift(1, fill_value=False)
        df_pair_win['run_id'] = (df_pair_win['is_exceed'] != df_pair_win['prev_exceed']).cumsum()
        # select only runs where is_exceed == True
        exceed_runs = (
            df_pair_win[df_pair_win['is_exceed']]
            .groupby('run_id', sort=False)
            .agg(
                segment_start_ts = ('timestamp','min'),
                segment_end_ts = ('timestamp','max'),
                segment_length = ('timestamp','count'),
                segment_max_value = ('value','max')
            )
            .reset_index(drop=True)
        )
        if exceed_runs.empty:
            longest_len = 0
            earliest_start_of_longest = pd.NA
            segment_max_value = pd.NA
            segment_max_dev = 0.0
        else:
            # pick longest segment, tie-breaker earliest start
            exceed_runs_sorted = exceed_runs.sort_values(by=['segment_length','segment_start_ts'], ascending=[False, True]).reset_index(drop=True)
            best = exceed_runs_sorted.iloc[0]
            longest_len = int(best['segment_length'])
            earliest_start_of_longest = int(best['segment_start_ts'])
            segment_max_value = float(best['segment_max_value'])
            # compute deviation ratio with p95==0 handling
            if pd.isna(global_p95):
                segment_max_dev = np.nan
            else:
                if global_p95 == 0:
                    segment_max_dev = np.inf if segment_max_value > 0 else 0.0
                else:
                    segment_max_dev = float((segment_max_value - global_p95) / global_p95)

    results.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': global_p95,
        'in_window_exceed_count': in_window_exceed_count,
        'in_window_exceed_timestamps_earliest20': exceed_ts_list,
        'longest_consecutive_exceed_segment_length': longest_len,
        'earliest_start_ts_of_longest_segment': earliest_start_of_longest,
        'segment_max_value': segment_max_value,
        'segment_max_deviation_ratio': segment_max_dev
    })

final_df = pd.DataFrame(results)

# Sort by earliest_start_ts_of_longest_segment ascending (NA last)
final_df['sort_key'] = final_df['earliest_start_ts_of_longest_segment'].apply(lambda x: x if pd.notna(x) else 10**18)
final_df_sorted = final_df.sort_values('sort_key').drop(columns=['sort_key']).reset_index(drop=True)

# Keep only requested columns and at most 5 rows
output = final_df_sorted[
    ['cmdb_id','kpi_name','global_p95','in_window_exceed_count','in_window_exceed_timestamps_earliest20',
     'longest_consecutive_exceed_segment_length','earliest_start_ts_of_longest_segment',
     'segment_max_value','segment_max_deviation_ratio']
].head(5)

output

```
Out[5]:
```
Summary (timestamps are Unix seconds UTC):

- catalogue | workload: global P95 = 3.513053. In-window exceedances = 2 at timestamps [1705652280, 1705653240]. Longest consecutive exceed segment length = 1 (earliest start 1705652280). Segment max = 4.645789, deviation ≈ 0.322436 (≈32.2% above P95).

- user | latency-50: global P95 = 0.005630. In-window exceedances = 2 at timestamps [1705652400, 1705653240]. Longest consecutive exceed segment length = 1 (earliest start 1705652400). Segment max = 0.005675, deviation ≈ 0.008003 (≈0.8% above P95).

- payment | latency-50: global P95 = 0.003000. In-window exceedances = 2 at timestamps [1705652400, 1705652880]. Longest consecutive exceed segment length = 1 (earliest start 1705652400). Segment max = 0.003651, deviation ≈ 0.217001 (≈21.7% above P95).

- user | socket: global P95 = 26.773333. In-window exceedances = 2 at timestamps [1705652400, 1705652460]. Longest consecutive exceed segment length = 2 (earliest start 1705652400). Segment max = 30.95, deviation ≈ 0.156001 (≈15.6% above P95).

- carts | latency-90: global P95 = 9.878108. In-window exceedances = 2 at timestamps [1705652460, 1705653480]. Longest consecutive exceed segment length = 1 (earliest start 1705652460). Segment max = 13.717732, deviation ≈ 0.388700 (≈38.9% above P95).

All five pairs had exceedances within the incident window; the most sustained run was user|socket (2 consecutive points). The largest relative deviations are carts|latency-90 and catalogue|workload.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  global_p95  in_window_exceed_count in_window_exceed_timestamps_earliest20  longest_consecutive_exceed_segment_length  earliest_start_ts_of_longest_segment  segment_max_value  segment_max_deviation_ratio
0  catalogue    workload    3.513053                       2               [1705652280, 1705653240]                                          1                            1705652280           4.645789                     0.322436
1       user  latency-50    0.005630                       2               [1705652400, 1705653240]                                          1                            1705652400           0.005675                     0.008003
2    payment  latency-50    0.003000                       2               [1705652400, 1705652880]                                          1                            1705652400           0.003651                     0.217001
3       user      socket   26.773333                       2               [1705652400, 1705652460]                                          2                            1705652400          30.950000                     0.156001
4      carts  latency-90    9.878108                       2               [1705652460, 1705653480]                                          1                            1705652460          13.717732                     0.388700```
```